# investigate data availability for various EBM data

In [ ]:
# Load paths for the SAPN EBM 4-5 dataset.
from ebm_all_paths.ebm_nsw_path1 import CIRCUIT_DETAILS_PATH, SITE_DETAILS_PATH, CLEANED_SITE_DATA_PATH
from loading import load_ebm_cleaned_data, load_ebm_circuit_details, load_ebm_site_details
import polars as pl

In [2]:
if not CLEANED_SITE_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned SAPN data not found: {CLEANED_SITE_DATA_PATH}\n"
        "Run sapn2022_workflow/run_sapn2022_preprocessing.py first."
    )
site_details    = load_ebm_site_details(SITE_DETAILS_PATH) # already filyres out single inverters
circuit_details = load_ebm_circuit_details(CIRCUIT_DETAILS_PATH)
all_data        = load_ebm_cleaned_data(CLEANED_SITE_DATA_PATH)

In [ ]:
# filter sites with 1-3 pv circuits and 1 inverter
PV_CIRCUIT_TYPES = ["pv_site_net", "pv_site"]

pv_circuit_details = circuit_details.filter(
    pl.col("con_type").is_in(PV_CIRCUIT_TYPES)
)
pv_circuit_counts = (
    pv_circuit_details
    .group_by("site_id")
    .len(name="pv_circuit_count")
)
# sites with 1 inverter and 1-3 PV circuits
candidate_site_ids = (
    all_data.filter(pl.col("con_type").is_in(PV_CIRCUIT_TYPES))
    .select("site_id")
    .unique()
    .join(site_details.select("site_id").unique().lazy(), on="site_id") # this filters out single inverter sites
    .join(pv_circuit_counts.lazy(), on="site_id")
    .filter(pl.col("pv_circuit_count").is_between(1, 3)) # filters out 1-3 pv circuits on a site
    .select("site_id")
    .collect()
    .get_column("site_id")
    .to_list()
)
len(candidate_site_ids)

1015

In [ ]:
# get rated capacity
# calculate later

In [4]:
# print(all_data.select(pl.col("local_tstamp").min()).collect())
# print(all_data.select(pl.col("local_tstamp").max()).collect())
# print(all_data.collect().head(10))

In [5]:
# single inverter filtering
# single_inverter_site_ids = (
#     pl.read_csv(SITE_DETAILS_PATH)
#     .filter(pl.col("inverter_count") == 1)
#     .get_column("site_id")
# )

# # filtering data for local timestamp
# all_data = all_data.filter(
#     pl.col("site_id").is_in(single_inverter_site_ids.implode()),
#     (pl.col("local_tstamp").dt.hour() >= 6)
#     & (pl.col("local_tstamp").dt.hour() < 18),
# )


In [ ]:
# prepapre each day for each site before feeding it in phase A
from datetime import datetime, time

from site_preparation import (
    calculate_site_day_voltage_signals,
    extract_site_day,
    map_circuit_data_to_site,
    select_site_pv_data,
    trim_site_day_analysis_window,
)

for site_id in candidate_site_ids:
    site_data = select_site_pv_data(all_data, circuit_details, site_id)

    local_dates = (
        site_data.select(pl.col("local_tstamp").dt.date().alias("local_date"))
        .drop_nulls()
        .unique()
        .sort("local_date")
        .get_column("local_date")
        .to_list()
    )

    for local_date in local_dates:
        site_day_long = extract_site_day(
            site_data,
            datetime.combine(local_date, time(5, 50)),
            datetime.combine(local_date, time(18, 0)),
        )
        site_day_wide = map_circuit_data_to_site(site_day_long, site_id)
        prepared_day = calculate_site_day_voltage_signals(site_day_wide)
        analysis_day = trim_site_day_analysis_window(
            prepared_day,
            time(6, 0),
            time(18, 0),
        )

In [6]:
# plotting should only beigin after you are comfortable with the method

In [ ]:
# Extracts each day from 05:50 through 18:00 local time.

# Maps the long circuit data into one wide row per timestamp.